## Building the Tiny GPT using the pytorch

---


### Defining the imports for Tiny GPT:

---


In [82]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import random

from transformer_block import Block 

print("Torch Version: ",torch.__version__)
device = 'cuda ' if torch.cuda.is_available() else 'cpu'
print(device)
print("GPU Name:",torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'cpu')

Torch Version:  2.5.1
cuda 
GPU Name: NVIDIA GeForce RTX 2050


### Defining the tiny dataset for trial :

---


In [83]:
corpus = [
    "hello friends how are you",
    "the tea is very hot",
    "my name is Aditya",
    "the roads of Delhi are busy",
    "it is raining in Mumbai",
    "the train is late again",
    "i love eating samosas and drinking tea",
    "holi is my favorite festival",
    "diwali brings lights and sweets",
    "india won the cricket match"
]


### Data pre-processing :

---


In [84]:
corpus = [s + " <END>" for s in corpus] ### putting the end in each sentences 

text= " ".join(corpus) ## joining the whole corpus 

print(text) 


hello friends how are you <END> the tea is very hot <END> my name is Aditya <END> the roads of Delhi are busy <END> it is raining in Mumbai <END> the train is late again <END> i love eating samosas and drinking tea <END> holi is my favorite festival <END> diwali brings lights and sweets <END> india won the cricket match <END>


### Convesion of words into Tokens without external tokenizers (testing purpose )

### modern LLM models are using the tokenizers

- gpt uses -> BPE(Byte pair encoding )
- Llama uses -> Sentence Piece

### but We dont use it now for case of simplicity:

---


In [85]:
words = list(set(text.split()))### which ever there is white space beten the words that word will considerd as token::(my logic )
print(words)

vocab_size = len(words) ## length of vocabuary->> no of unique words 
print(vocab_size)

['and', 'samosas', 'how', 'brings', 'are', 'the', 'won', 'love', 'india', 'you', 'sweets', 'Delhi', 'is', 'my', 'late', 'i', 'drinking', 'holi', 'cricket', 'eating', 'favorite', 'train', 'match', 'friends', 'lights', 'of', 'raining', 'very', 'festival', 'tea', 'hello', 'Mumbai', 'name', 'it', 'Aditya', 'busy', 'diwali', 'roads', 'hot', '<END>', 'again', 'in']
42


### Conversion of words into numbers(tensors) or words to indexes --->> (Manually )

---


In [86]:
word2idx = {w: i for i,w in enumerate(words)}
print("word2idx: ",word2idx)

word2idx:  {'and': 0, 'samosas': 1, 'how': 2, 'brings': 3, 'are': 4, 'the': 5, 'won': 6, 'love': 7, 'india': 8, 'you': 9, 'sweets': 10, 'Delhi': 11, 'is': 12, 'my': 13, 'late': 14, 'i': 15, 'drinking': 16, 'holi': 17, 'cricket': 18, 'eating': 19, 'favorite': 20, 'train': 21, 'match': 22, 'friends': 23, 'lights': 24, 'of': 25, 'raining': 26, 'very': 27, 'festival': 28, 'tea': 29, 'hello': 30, 'Mumbai': 31, 'name': 32, 'it': 33, 'Aditya': 34, 'busy': 35, 'diwali': 36, 'roads': 37, 'hot': 38, '<END>': 39, 'again': 40, 'in': 41}


### Conversion of indexes into word --->> (opposite of above step)

---


In [87]:
idx2words = {i: w for w,i in word2idx.items()}
print("idx2word: ",idx2words)

idx2word:  {0: 'and', 1: 'samosas', 2: 'how', 3: 'brings', 4: 'are', 5: 'the', 6: 'won', 7: 'love', 8: 'india', 9: 'you', 10: 'sweets', 11: 'Delhi', 12: 'is', 13: 'my', 14: 'late', 15: 'i', 16: 'drinking', 17: 'holi', 18: 'cricket', 19: 'eating', 20: 'favorite', 21: 'train', 22: 'match', 23: 'friends', 24: 'lights', 25: 'of', 26: 'raining', 27: 'very', 28: 'festival', 29: 'tea', 30: 'hello', 31: 'Mumbai', 32: 'name', 33: 'it', 34: 'Aditya', 35: 'busy', 36: 'diwali', 37: 'roads', 38: 'hot', 39: '<END>', 40: 'again', 41: 'in'}


### Conversion of Numbers into Tensors :

---


In [88]:
data = torch.tensor([word2idx[w] for w in text.split()], dtype= torch.long)
print(data)

tensor([30, 23,  2,  4,  9, 39,  5, 29, 12, 27, 38, 39, 13, 32, 12, 34, 39,  5,
        37, 25, 11,  4, 35, 39, 33, 12, 26, 41, 31, 39,  5, 21, 12, 14, 40, 39,
        15,  7, 19,  1,  0, 16, 29, 39, 17, 12, 13, 20, 28, 39, 36,  3, 24,  0,
        10, 39,  8,  6,  5, 18, 22, 39])


In [89]:
print(len(data))

62


### Creating the batches for dataset :

- context window : context length means how much the data or tokens that llm can see before generating the response
- embeddimg dim : means that your model have that N no of values for each word
- ***


In [90]:
from torch import embedding

## hyper-parameters
block_size =6 #  here context window for testing 
embedding_dim = 32 ## 
n_heads = 2 ## no of self attention heads 
n_layers = 2 ## no of transformer block 
lr = 1e-3 ## learning rate 
epochs = 1500
batch_size = 16 ## no of sequences 



In [91]:
## get batch function 

def get_batch(batch_size=16):
    ix = torch.randint(len(data)- block_size,(batch_size,)) ##  data = 62 - block_size+ = 6 = 56 (0 -> 55 )indexex  ,16
# [12, 32,45 ..... 4 ] those 16 random indexes we give to the model  in the batch  every sequence has 6 random samples 

    x = torch.stack([data[i:i+block_size] for i in ix]) ## if X = [t12 , t13,t14 ,t15,t16,t17] 6 tokean samples 
    y = torch.stack([data[i+1:i+block_size +1 ] for i in ix])  ## if y = [t13 , t14,t15 ,t16,t17,t18] 6 tokean samples 
    return x,y
    

## Tiny GPT class :

---


In [92]:
class TinyGPT(nn.Module):
    def __init__(self):
        super().__init__()
        self.token_embedding = nn.Embedding(vocab_size, embedding_dim)    # (42,32)

        self.position_embedding = nn.Embedding(block_size,embedding_dim) ## orders of the words (6,32)
        self.block = nn.Sequential(*[Block(embedding_dim,block_size,n_heads) for _ in range(n_layers)])

        self.ln_f = nn.LayerNorm(embedding_dim)

        self.head = nn.Linear(embedding_dim,vocab_size)
    def forward(self, idx, targets=None):
        B, T = idx.shape ## B = Batch size ||||  T = Conext length (16,6)
        tok_emb = self.token_embedding(idx) 
        
        pos_emb = self.position_embedding(torch.arange(T, device=idx.device))
        x = tok_emb + pos_emb  
        x = self.block(x) 
        x = self.ln_f(x)
        logits = self.head(x) 
        loss = None
        if targets is not None:
            B, T, C = logits.shape 
            loss = F.cross_entropy(logits.view(B*T, C), targets.view(B*T)) 
        return logits, loss

    def generate(self, idx, max_new_tokens): ### generate next word function 
        for _ in range(max_new_tokens):
            idx_cond = idx[:, -block_size:]
            logits, _ = self(idx_cond)
            logits = logits[:, -1, :]
            probs = F.softmax(logits, dim=-1)
            next_idx = torch.multinomial(probs, 1)
            idx = torch.cat((idx, next_idx), dim=1)
        return idx





## Creating the model:

---


In [93]:
model = TinyGPT()
optimizer = torch.optim.AdamW(model.parameters(), lr=lr) 



### Training :

---


In [101]:
model.eval()

for step in range(epochs):
    xb, yb = get_batch() 
    logits, loss = model(xb, yb)
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    if step % 300 == 0:
        print(f"Step {step}, loss={loss.item():.4f}")



Step 0, loss=0.1119
Step 300, loss=0.1335
Step 600, loss=0.0900
Step 900, loss=0.0973
Step 1200, loss=0.1212


## Testing the Model:

---


In [103]:
context = torch.tensor([[word2idx["hello"]]], dtype=torch.long)
out = model.generate(context, max_new_tokens=15)

print("\nGenerated text:\n")
print(" ".join(idx2words[int(i)] for i in out[0]))


Generated text:

hello friends how are you <END> the tea is very hot <END> my name is Aditya
